# Education Outflow Prediction
This notebook loads the dbt-created DuckDB dataset and trains a simple prediction model using education-based migration features.

In [1]:
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import joblib

In [7]:
con = duckdb.connect('../duckdb/brazil_migration.duckdb')

query = '''
select year, education_group, total_migrants, male_ratio, female_ratio
from analytics_marts.dim_brazil_education_features
'''

data = con.execute(query).df()

In [8]:
data = data.dropna(subset=['total_migrants'])
data['education_group'] = data['education_group'].astype('category').cat.codes
X = data[['education_group', 'male_ratio', 'female_ratio']]
y = data['total_migrants']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
joblib.dump(model, '../models/prediction_model.pkl')

['../models/prediction_model.pkl']

## Quick prediction example
Use the trained model to predict the next total migrant count for a high-education group.

In [5]:
sample = pd.DataFrame([{'education_group': 1, 'male_ratio': 0.5, 'female_ratio': 0.5}])
print('Prediction for sample:', model.predict(sample))

Prediction for sample: [340334.76]


In [14]:
con = duckdb.connect('../duckdb/brazil_migration.duckdb')

query = '''
with education_outflow as (
    select
        year,
        education_group,
        total_migrants,
        lag(total_migrants) over (partition by education_group order by year) as previous_year_total,
        case
            when lag(total_migrants) over (partition by education_group order by year) is null then null
            else total_migrants - lag(total_migrants) over (partition by education_group order by year)
        end as year_delta
    from analytics_marts.dim_brazil_education_features
)

select
    year,
    education_group,
    total_migrants,
    previous_year_total,
    year_delta,
    case
        when previous_year_total is null then null
        else year_delta / previous_year_total
    end as growth_rate,
    case
        when education_group = 'high education' then 1
        else 0
    end as high_education_flag
from education_outflow
order by year desc, growth_rate desc
'''

data = con.execute(query).df()

In [15]:
print(data)

    year education_group  total_migrants  previous_year_total  year_delta  \
0   2024               B        990307.0             790102.0    200205.0   
1   2024             B R       1066929.0             904419.0    162510.0   
2   2024         unknown      15757595.0           13418219.0   2339376.0   
3   2024             C R         11466.0              10418.0      1048.0   
4   2024           C B R            12.0                 12.0         0.0   
5   2024               C        207252.0             211743.0     -4491.0   
6   2020               B        790102.0             565015.0    225087.0   
7   2020             B R        904419.0             684930.0    219489.0   
8   2020         unknown      13418219.0           10365776.0   3052443.0   
9   2020               C        211743.0             176152.0     35591.0   
10  2020             C R         10418.0               9133.0      1285.0   
11  2020           C B R            12.0                 14.0        -2.0   